# 项目二 Kaggle 冒烟测试

本 Notebook 只做环境和最小训练验证，不启动正式训练。数据目录应包含已重定向到 G1 的 CSV，并确认数据已通过许可证核查。

本版把想象 rollout 的 policy batch 从 4096 降到 256，只改这一个变量；GPU 0 单卡训练，双卡会话用于后续并行两个实验。
> 已停止作为主路线（双 T4 上默认 batch 4096 反向传播 OOM）。保留作失败证据；主路线见 `kaggle_blinkmimic_t4.ipynb`。


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path('/kaggle/working/proj2-humanoid-mimic')
UPSTREAM_DIR = PROJECT_ROOT / 'third_party' / 'SuperMimic-Reproduced'
UPSTREAM_URL = 'https://github.com/CuboiLeo/SuperMimic-Reproduced.git'
UPSTREAM_COMMIT = '3602aed6a397b3202ac052d688d47300f443a3ea'
DATA_DIR = Path('/kaggle/input/datasets/machineabao/smoke1')
INPUT_FPS = 30.0
OUTPUT_FPS = 50.0
NUM_ENVS = 1
POLICY_BATCH = 256
TOTAL_ITERATIONS = 100
RUN_NAME = 'kaggle_smoke'
LOG_DIR = PROJECT_ROOT / 'results' / RUN_NAME
SMOKE_TIMEOUT_SECONDS = 1800

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_DIR={DATA_DIR}')
print(f'NUM_ENVS={NUM_ENVS} POLICY_BATCH={POLICY_BATCH} TOTAL_ITERATIONS={TOTAL_ITERATIONS}')


In [ ]:
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(str(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'])
gpu_count = int(subprocess.check_output(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'], text=True).count('\n'))
print(f'gpu_count={gpu_count}')
if gpu_count != 2:
    raise RuntimeError(f'需要恰好 2 张 GPU，当前检测到 {gpu_count} 张')

In [ ]:
import shutil

(PROJECT_ROOT / 'third_party').mkdir(parents=True, exist_ok=True)
if not (UPSTREAM_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', UPSTREAM_URL, str(UPSTREAM_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], cwd=UPSTREAM_DIR)
run(['git', 'checkout', '--detach', UPSTREAM_COMMIT], cwd=UPSTREAM_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=UPSTREAM_DIR, text=True).strip()
print(f'git_head={head}')
if head != UPSTREAM_COMMIT:
    raise RuntimeError('上游 commit 校验失败')

run(['python', '-V'], cwd=UPSTREAM_DIR)
run(['python', '-m', 'pip', 'install', '--upgrade', 'pip', 'uv'], cwd=UPSTREAM_DIR)
run(['uv', 'sync', '--frozen', '--no-dev'], cwd=UPSTREAM_DIR)

In [ ]:
version_check = '''
import importlib.metadata as metadata
import sys
import torch
print(f'python={sys.version.split()[0]}')
print(f'torch={torch.__version__}')
print(f'torch_cuda={torch.version.cuda}')
print(f'cuda_available={torch.cuda.is_available()}')
print(f'cuda_device_count={torch.cuda.device_count()}')
for name in ('mjlab', 'mujoco', 'mujoco-warp', 'warp-lang'):
    try:
        print(f'{name}={metadata.version(name)}')
    except metadata.PackageNotFoundError:
        print(f'{name}=NOT_FOUND')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA 不可用')
'''
run(['uv', 'run', 'python', '-c', version_check], cwd=UPSTREAM_DIR)
with open(LOG_DIR / 'train_help.txt', 'w') as output:
    subprocess.run(['uv', 'run', 'python', '-m', 'supermimic.scripts.train', '--help'], cwd=UPSTREAM_DIR, stdout=output, stderr=subprocess.STDOUT, check=True)
print('train_help=PASS')

In [ ]:
import shutil

if not DATA_DIR.exists():
    raise FileNotFoundError(f'动作数据目录不存在: {DATA_DIR}')
csv_files = sorted(DATA_DIR.glob('*.csv'))[:2]
if not csv_files:
    raise FileNotFoundError(f'动作数据目录没有 CSV 文件: {DATA_DIR}')
smoke_csv_dir = PROJECT_ROOT / 'data' / 'smoke_csv'
smoke_data_dir = PROJECT_ROOT / 'data' / 'smoke_npz'
smoke_csv_dir.mkdir(parents=True, exist_ok=True)
smoke_data_dir.mkdir(parents=True, exist_ok=True)
for source in csv_files:
    target = smoke_csv_dir / source.name
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
print('selected_csv=')
for item in csv_files:
    print(item)

convert_command = [
    'uv', 'run', 'python', 'scripts/data/csv_to_npz_batch.py',
    '--input-dir', str(smoke_csv_dir),
    '--output-dir', str(smoke_data_dir),
    '--input-fps', str(INPUT_FPS),
    '--output-fps', str(OUTPUT_FPS),
    '--workers', '1',
    '--limit', str(len(csv_files)),
]
run(convert_command, cwd=UPSTREAM_DIR)
npz_files = sorted(smoke_data_dir.glob('*.npz'))
if not npz_files:
    raise RuntimeError('CSV 转换结束但未生成 NPZ')
print(f'converted_npz_count={len(npz_files)}')

In [ ]:
import csv
import threading
import time

env = os.environ.copy()
env['PYTHONPATH'] = str(UPSTREAM_DIR / 'src') + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
env['MUJOCO_GL'] = 'egl'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gpu_log = LOG_DIR / 'gpu_samples.csv'
gpu_log.parent.mkdir(parents=True, exist_ok=True)
stop_monitor = threading.Event()

def monitor_gpu():
    with open(gpu_log, 'w', newline='') as output:
        writer = csv.writer(output)
        writer.writerow(['timestamp', 'gpu_index', 'memory_used_mib', 'utilization_pct'])
        while not stop_monitor.is_set():
            result = subprocess.run(['nvidia-smi', '--query-gpu=timestamp,index,memory.used,utilization.gpu', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False)
            for line in result.stdout.splitlines():
                writer.writerow([part.strip() for part in line.split(',')])
            output.flush()
            stop_monitor.wait(2)

monitor = threading.Thread(target=monitor_gpu, daemon=True)
monitor.start()
command = [
    'uv', 'run', 'sm-train',
    '--run-name', RUN_NAME,
    '--log-root', str(LOG_DIR / 'logs'),
    '--data.npz-dir', str(smoke_data_dir),
    '--data.limit', str(len(npz_files)),
    '--env.num-envs', str(NUM_ENVS),
    '--policy-trainer.batch-size', str(POLICY_BATCH),
    '--schedule.total-iterations', str(TOTAL_ITERATIONS),
    '--schedule.warmup-iterations', '40',
    '--schedule.log-interval', '10',
    '--schedule.checkpoint-interval', '100',
    '--schedule.max-hours', '0.5',
]
print('$', ' '.join(command))
started = time.time()
try:
    result = subprocess.run(command, cwd=UPSTREAM_DIR, env=env, timeout=SMOKE_TIMEOUT_SECONDS)
finally:
    stop_monitor.set()
    monitor.join(timeout=5)

print(f'train_wall_seconds={time.time() - started:.1f}')
rows = list(csv.DictReader(open(gpu_log)))
for gpu_index in sorted({row['gpu_index'] for row in rows}):
    values = [int(row['memory_used_mib']) for row in rows if row['gpu_index'] == gpu_index]
    print(f'gpu_{gpu_index}_peak_memory_mib={max(values) if values else 0}')

if result.returncode != 0:
    raise RuntimeError(f'训练失败，exit_code={result.returncode}，请查看 {LOG_DIR}')
print('status=PASS_100_STEP_SMOKE')
